# Lab 05: How to rebuild a RAG pipeline with LangChain

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Demo**

## Goal

Rebuild our plain Python RAG pipeline with LangChain and check which behaviour is preserved. We will compare retrieval results question by question, generate cited answers from shared evidence, and use metadata to scope a customer's search.

The first comparison shows why this matters: both searches find a labelled source for every development question, yet their complete result lists differ because one returns an extra chunk. A total alone hides the default that changed the evidence.

We already know the pipeline's steps from Labs 01–04. LangChain provides components with common interfaces for documents, embeddings, vector stores and chat models. Reusing these components can simplify the code we maintain, but we still need to understand their inputs, outputs and defaults.

Parts A–D rebuild and compare retrieval. Parts E–H continue with cited answers, product filtering and the reusable pipeline that the next labs evaluate.

## What you will learn

- How to turn chunk records into LangChain documents that keep their IDs and fields.
- How LangChain's embeddings and chat model are built from `models.env`, and what the embeddings object sends to the
  API.
- How to index documents in a vector store and search them with scores.
- How to compare a framework's results with your own question by question, and find the default that made them
  differ.
- How to get a cited answer back as a Python object, check it with Lab 02's quote check, and explain why its answers
  can differ from your own.
- How to leave documents out of a search with a metadata filter.

**In class:** Part A, Part B through Checkpoint B, then Parts C and D, shown as a demo. Part B first uses
`OpenAIEmbeddings` directly, then introduces the shared helper. The remaining embedding comparison in Part B and
Parts E to H are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-28
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` and `TOKEN_ENCODING` from
`models.env`. It makes about 110 short chat calls and embeds the corpus two or three times (about 21,000 tokens each),
well under one US cent on `gpt-4o-mini`. The saved outputs and every count in the text come from `gpt-4o-mini`; on
another model the counts can move. Model replies vary from run to run and similarity scores can move in the
third decimal, so compare the order of results and the kind of reply with the text, not every number.

In the printed output, *plain* is your pipeline from Labs 01 and 02 and *LangChain* the new one.

---

## Part A: Represent Our Chunks as LangChain Documents

So far, we've represented chunks as Python dictionaries. LangChain uses a `Document` object to represent text along with its metadata. In this section, we'll convert our existing chunks into __LangChain Documents__ without changing their content.

**Why do we need this conversion?** LangChain's vector store accepts `Document` objects when we call `add_documents()`. Each object will carry both the text we want to search and the information we need to identify its source afterwards.

We are keeping the same corpus as Lab 02: 272 SQuAD chunks and five product policies. Each policy is one searchable record, with a `chunk_id` equal to its `source_id`. This lets us compare the two implementations using the same starting data.


In [3]:
# corpus: 277 records, the 272 chunks followed by the five policies; each record is a dictionary
import json

DATA = ROOT / "data"
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus = chunks + [{**policy, "chunk_id": policy["source_id"]} for policy in policies]
print(len(corpus), "records")
print("a chunk: ", list(corpus[0]))
print("a policy:", list(corpus[-1]))

277 records
a chunk:  ['chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']
a policy: ['source_id', 'title', 'text', 'version', 'access', 'chunk_id']


We now have **277 records**, each represented as a dictionary:

- A SQuAD chunk includes its text, `chunk_id`, source paragraph (`source_id`), character offsets (`start`, `end`) and token count.
- A policy includes fields such as `version` and `access`. `{**policy, "chunk_id": ...}` copies its existing fields and adds the chunk identifier our pipeline expects.

### How does a dictionary become a Document?

The [`Document`](https://reference.langchain.com/python/langchain-core/documents/base/Document) object keeps these values under three attributes:

| Document attribute | Value from our record | What we use it for |
|---|---|---|
| `page_content` | `record["text"]` | Text to embed and later include in the evidence |
| `metadata` | All fields except `text` | Source checks, filtering and information about the record |
| `id` | `record["chunk_id"]` | Identify the stored chunk and cite it in an answer |

For example, the first chunk's ID is `S00-P00-C00`, while its source paragraph is `S00-P00`. We keep both: a citation can identify the particular chunk, and a retrieval check can recognise the paragraph it came from.

First inspect that original record. Then create one `Document` from it and print the three attributes so we can see where the values went.


In [4]:
# show the first record in the corpus
corpus[0]

{'chunk_id': 'S00-P00-C00',
 'source_id': 'S00-P00',
 'title': 'Super_Bowl_50',
 'start': 0,
 'end': 486,
 'text': "Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the",
 'tokens': 98}

In [5]:
from langchain_core.documents import Document

record = corpus[0]

# document: the first record as a Document; the text goes to page_content, every other field to metadata
document = Document(id=record["chunk_id"], page_content=record["text"],
                    metadata={name: value for name, value in record.items() if name != "text"})
print("id:      ", document.id)
print("metadata:", document.metadata)
print("text:    ", document.page_content[:80], "...")

id:       S00-P00-C00
metadata: {'chunk_id': 'S00-P00-C00', 'source_id': 'S00-P00', 'title': 'Super_Bowl_50', 'start': 0, 'end': 486, 'tokens': 98}
text:     Super Bowl 50 was an American football game to determine the champion of the Nat ...


The original text is now in `document.page_content`. Its ID is `document.id`, and the remaining fields are in `document.metadata`.

The expression `{name: value for name, value in record.items() if name != "text"}` builds that metadata dictionary. It goes through every field, keeping the name and value unless the field is `text`.

__Why keep the original ID?__ The store can generate an ID when one is missing, but we already have identifiers tied to our sources. Reusing them lets us compare results and check citations against the same chunks in both pipelines.

__Does the model automatically receive the metadata?__ No. In this lab, the store embeds `page_content`. Later, our `evidence()` function decides which text and IDs to send to the chat model. Fields such as `access` remain available to our code, but storing them does not automatically enforce a permission rule.

Now repeat the conversion for all 277 records. The list comprehension applies the same `Document(...)` constructor to each record in `corpus`, preserving their order.


In [6]:
# documents: one Document per record, built the same way as the one above
documents = [Document(id=record["chunk_id"], page_content=record["text"],
                      metadata={name: value for name, value in record.items() if name != "text"})
             for record in corpus]
print(len(documents), "documents")

# print the last document to show that it is a policy, not a chunk
print(documents[-1])

277 documents
page_content='Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.' metadata={'source_id': 'H05', 'title': 'Atlas billing', 'version': '2026-10-01', 'access': 'public', 'chunk_id': 'H05'}


The last object, `documents[-1]`, is policy **H05**. Its policy text is in `page_content`; its title, `source_id`, `version`, `access` and `chunk_id` are still available in `metadata`.

We have changed the representation, while keeping the text and source information. This is what makes the next comparison possible: LangChain will search the same content that our plain Python pipeline searches.

### Checkpoint A

| You should see | Expected structure |
|---|---|
| Corpus | 277 records: 272 chunks and five policies |
| Documents | One `Document` per record, with the original text and ID |
| Last document | H05, with `version` and `access: 'public'` in its metadata |

Next, we need an embeddings object that the vector store can call when it receives these documents.


---

## Part B: Configure LangChain Embeddings and Compare Their Vectors

In Lab 01, we called the embedding API through our own client. Now we will do the same work directly with LangChain's `OpenAIEmbeddings` class before moving the configuration into a helper.

We will build an embeddings object, embed one question, then embed a list of short texts. This shows the methods that Part C's vector store will call for us.

### Create an OpenAIEmbeddings object

Import `OpenAIEmbeddings` from `langchain_openai` and construct it with the model, endpoint and key from our existing configuration. The class calls the embedding API; it does not load an embedding model onto our laptop.

- `model` selects `EMBED_MODEL`, the same model as Lab 01.
- `base_url` selects the configured embedding endpoint, using `embed_url()` from our settings module.
- `api_key` supplies the key loaded by Config. The small key-selection block also supports a separately configured embedding provider.
- `check_embedding_ctx_length` enables LangChain's tokenization and long-input handling for OpenAI; we turn it off for other endpoints so LangChain sends raw text.

Constructing this object does not embed any text yet. We will make the first request in the next step.


In [7]:
import os
from langchain_openai import OpenAIEmbeddings
from shared.settings import embed_url, OPENAI_DEFAULT_URL

embedding_key = os.environ["OPENAI_API_KEY"]
if settings.get("EMBED_BASE_URL") and load_key("EMBED_API_KEY"):
    embedding_key = os.environ["EMBED_API_KEY"]

embeddings = OpenAIEmbeddings(
    model=settings["EMBED_MODEL"],
    base_url=embed_url(settings),
    api_key=embedding_key,
    check_embedding_ctx_length=embed_url(settings) == OPENAI_DEFAULT_URL,
)
print(type(embeddings).__name__, "| model:", embeddings.model)


OpenAIEmbeddings | model: text-embedding-3-small


### Embed one question

`embeddings` is now an `OpenAIEmbeddings` object. Its `embed_query()` method accepts one string and returns one vector as a Python list of floating-point numbers.

Let's use a familiar Atlas question. We will print the vector's type, its length and the first five values, rather than display all its numbers.


In [8]:
example_question = "How long does an Atlas reset link remain valid?"
example_vector = embeddings.embed_query(example_question)

print("type:", type(example_vector).__name__)
print("dimensions:", len(example_vector))
print("first five values:", example_vector[:5])


type: list
dimensions: 1536
first five values: [-0.034393310546875, -0.01021575927734375, 0.006702423095703125, -0.0110015869140625, -0.0521240234375]


The result is one vector: **1,536 values with the default embedding model**. Individual values do not give us an answer or a confidence score. They are the numerical representation used to compare this question with document vectors.

### Embed several texts

For a list of texts, use `embed_documents()`. It returns a list of vectors, one per input text, in the same order.

We will embed two short policy statements: one about Atlas and one about Boreal. These are plain strings; this method does not require `Document` objects. Part C's store will extract `page_content` from our Documents and pass those texts to the embeddings object.


In [9]:
example_texts = [
    "Atlas reset links expire after 30 minutes.",
    "Boreal reset links expire after 10 minutes.",
]
example_vectors = embeddings.embed_documents(example_texts)

print("texts:", len(example_texts), "| vectors:", len(example_vectors))
for text, vector in zip(example_texts, example_vectors):
    print(text, "| dimensions:", len(vector))


texts: 2 | vectors: 2
Atlas reset links expire after 30 minutes. | dimensions: 1536
Boreal reset links expire after 10 minutes. | dimensions: 1536


We now have **two vectors**, each with 1,536 values under the default configuration. `example_vectors[0]` belongs to the Atlas statement and `example_vectors[1]` to the Boreal statement.

The two methods serve the same roles we built in Lab 01:

| Method | Input | Output | Where the store uses it |
|---|---|---|---|
| `embed_query()` | One string | One vector | Embed a search question |
| `embed_documents()` | A list of strings | One vector per string | Embed document text when indexing |

### Reuse this configuration with lc_embeddings()

We could repeat the `OpenAIEmbeddings(...)` constructor in every lab. Instead, `lc_embeddings()` in `shared/settings.py` keeps the model, endpoint, key selection and tokenization setting in one place.

The helper returns the same kind of `OpenAIEmbeddings` object we just used. It does not embed texts or introduce another embedding algorithm. Its returned object still exposes `embed_query()` and `embed_documents()`.

Now print the helper's source and replace our directly constructed `embeddings` with the object it returns. From this point onward, we can use the helper without repeating the configuration block.


In [10]:
# embeddings: the LangChain object that embeds texts with EMBED_MODEL; Part C's store embeds with it
import inspect
from shared.settings import lc_embeddings

print(inspect.getsource(lc_embeddings))
embeddings = lc_embeddings(settings)
print("send token IDs:", embeddings.check_embedding_ctx_length)

def lc_embeddings(settings: dict | None = None):
    """The same embedding model as embed_client(), as LangChain OpenAIEmbeddings (Lab 05 on).

    LangChain splits long texts with tiktoken and sends token IDs by default; other OpenAI-compatible servers
    may reject token IDs, so that is switched off whenever the endpoint is not OpenAI's.
    """
    from langchain_openai import OpenAIEmbeddings
    settings = settings or load_settings()
    key = os.environ.get("OPENAI_API_KEY")
    if settings.get("EMBED_BASE_URL") and load_key("EMBED_API_KEY"):
        key = os.environ["EMBED_API_KEY"]
    return OpenAIEmbeddings(model=settings["EMBED_MODEL"], base_url=embed_url(settings), api_key=key,
                            check_embedding_ctx_length=embed_url(settings) == OPENAI_DEFAULT_URL)

send token IDs: True


### What did the helper configure?

`inspect.getsource(lc_embeddings)` prints the function we just called. Its return statement constructs `OpenAIEmbeddings` with:

- `model`: the configured `EMBED_MODEL`.
- `base_url`: the embedding endpoint printed in Config.
- `api_key`: the key selected for that endpoint. The helper can use a separate embedding key when configured.
- `check_embedding_ctx_length`: whether LangChain tokenizes inputs and handles texts longer than the model's embedding limit.

In the OpenAI configuration used for the reference run, that last option is `True`: LangChain sends token IDs to the embedding API. Our helper sets it to `False` for other endpoints so it sends raw text, which those providers are more likely to accept.

The helper has replaced our manual constructor with the same configuration steps. We have embedded only the small examples so far; the 277 corpus documents will be embedded when we add them to the store.

### Checkpoint B

| You should see | Reference configuration |
|---|---|
| Direct API usage | `OpenAIEmbeddings`, one question vector and two document vectors |
| Shared configuration | `lc_embeddings(settings)` returns another `OpenAIEmbeddings` object |
| Tokenization option | `send token IDs: True` for the default OpenAI endpoint |

### Compare one embedding with our existing function (self-study)

Before trusting the wrapper, let's give it a question we already know: *In what year was Nikola Tesla born?*

We will embed that question with both `embeddings.embed_query()` and Lab 01's `embed()`, then compare corresponding numbers. If the wrapper is using the same model and input, the vectors should be very close.


In [11]:
# lc_vector: the question embedded through LangChain; plain_vector: the same question through Lab 01's embed()
import numpy as np
from shared.search import embed

question = "In what year was Nikola Tesla born?"
lc_vector = np.array(embeddings.embed_query(question))
plain_vector = embed([question], settings)[0]
print("numbers:", len(lc_vector), "| largest difference from embed():", f"{np.abs(lc_vector - plain_vector).max():.6f}")

numbers: 1536 | largest difference from embed(): 0.000000


The reference comparison produced **1,536 numbers** in each vector, with a largest displayed difference of `0.000000`. Small differences are possible between API calls, so do not require exact equality.

### How are we comparing the vectors?

1. `embed_query(question)` returns one list of embedding values. `np.array(...)` turns it into a NumPy array so we can subtract the vectors.
2. `embed([question], settings)` returns one row per input text. `[0]` selects the row for our question.
3. Subtracting the arrays compares values at corresponding positions.
4. `np.abs(...).max()` finds the largest absolute difference; the print formats it to six decimal places.

### What was sent to the API?

The helper's `check_embedding_ctx_length=True` setting means LangChain tokenizes the question before sending it. A **token ID** is a number identifying a piece of text in the model's vocabulary; it is different from a value in the embedding vector returned by the API.

The next cell uses `cl100k_base`, the encoding used here for the OpenAI embedding model, to show those input IDs. This is separate from the `TOKEN_ENCODING` setting used later to count evidence tokens.


In [12]:
# token_ids: the question as the list of numbers LangChain sends in place of the text
import tiktoken

token_ids = tiktoken.get_encoding("cl100k_base").encode(question)
print("text:     ", question)
print("token IDs:", token_ids)

text:      In what year was Nikola Tesla born?
token IDs: [644, 1148, 1060, 574, 97415, 28298, 9405, 30]


The Tesla question becomes **eight token IDs** in this encoding:

`[644, 1148, 1060, 574, 97415, 28298, 9405, 30]`

These eight integers represent the input text. The 1,536 embedding values represent the model's output for that text. They have different jobs and different lengths.

The API can accept the text's token IDs directly, which explains how the request representation can change while the resulting embedding remains close to Lab 01's. For other endpoints, our helper sends the text instead.

| Self-study check | Reference result |
|---|---|
| Output embedding | 1,536 values; largest displayed difference from `embed()` is `0.000000` |
| Input representation | Eight token IDs for this question |

We now have an embeddings object configured for the same model as our original search. Next, we will let a vector store manage the documents and their vectors.


---

## Part C: Store Document Embeddings and Retrieve Similar Chunks

In Lab 01, we kept chunk vectors in a NumPy matrix and kept their IDs aligned with the rows. A **vector store** manages that relationship for us: each stored record connects a document, its ID and metadata, and its embedding.

We will use [`InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore). It keeps its data in the running Python process and ranks documents by cosine similarity. Restarting the kernel removes this in-memory store; we can rebuild it from our documents.

### What happens when we add the documents?

`InMemoryVectorStore(embeddings)` creates an empty store configured to use Part B's embeddings object. `store.add_documents(documents)` then embeds each document's `page_content` and stores the vectors alongside their documents.

The returned `ids` list tells us which identifiers the store used. We will print the first and last IDs to check that the original chunk and policy IDs were kept.


In [13]:
# store: the 277 documents with their vectors; ids: the ID each document was filed under, in order
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(embeddings)
ids = store.add_documents(documents)
print(len(ids), "documents stored | first IDs:", ids[:2], "| last:", ids[-2:])

277 documents stored | first IDs: ['S00-P00-C00', 'S00-P00-C01'] | last: ['H04', 'H05']


The store contains **277 documents** and has kept the original IDs, including `S00-P00-C00` and H05.

Now we can compare one search with our existing implementation. We will ask the Tesla question through both:

| Search | What it uses |
|---|---|
| `search()` from Lab 01 | Our corpus dictionaries and cached chunk vectors |
| `store.similarity_search_with_score()` | The documents and embeddings managed by the LangChain store |

The store embeds the query, compares it with the stored document vectors, and returns the closest documents with scores. We will initially call both searches without specifying `k`, the number of results, so we can inspect what their defaults do.

Before running the cell, predict whether the number of returned rows and the first three chunk IDs will match. Keep the row count separate from the similarity scores when you read the output.


In [14]:
# vectors: one cached vector per record; each search prints its hits as score and chunk ID, best first
from shared.search import chunk_vectors, search

question = "In what year was Nikola Tesla born?"
vectors = chunk_vectors(corpus, settings)
print("plain: search()")
for score, chunk in search(question, corpus, vectors, settings):
    print(f"  {score:.4f}  {chunk['chunk_id']}")
print("LangChain: store.similarity_search_with_score()")
for document, score in store.similarity_search_with_score(question):
    print(f"  {score:.4f}  {document.id}")

plain: search()


  0.7222  S03-P00-C00
  0.5405  S03-P01-C00
  0.4657  S03-P02-C00
LangChain: store.similarity_search_with_score()


  0.7221  S03-P00-C00
  0.5405  S03-P01-C00
  0.4656  S03-P02-C00
  0.2592  S03-P01-C01


### Read the returned objects

The two search functions package their results differently:

- Our `search()` returns `(score, chunk_dictionary)` pairs.
- The LangChain store returns `(Document, score)` pairs.

That is why the two loops unpack their variables in opposite orders, and why one reads `chunk["chunk_id"]` while the other reads `document.id`.

In the reference run, the same three chunks came first: `S03-P00-C00`, `S03-P01-C00` and `S03-P02-C00`. Their scores were very close. The store embedded the documents afresh, while our search used cached vectors, so small numerical differences are expected.

However, **the LangChain search returned four rows and our search returned three**. Both found the Tesla passage, but they did not return identical result lists. Part D checks whether an overall retrieval score reveals this difference.


For this store, the printed score is **cosine similarity**: a larger value means a closer match. Other vector-store methods may return a distance, where a smaller value means closer. Read the chosen method's definition before comparing its scores with ours.

### Checkpoint C

| You should see | Reference run |
|---|---|
| Stored documents | 277, with their original IDs |
| First three Tesla hits | `S03-P00-C00` about 0.72, `S03-P01-C00` about 0.54, `S03-P02-C00` about 0.47 |
| Row counts | Three from `search()`, four from the LangChain store |
| Extra LangChain hit | `S03-P01-C01`, about 0.26 |

Compare the IDs and order first. The final decimals can vary between embedding calls.


---

## Part D: Compare Retrieval Results and Check Search Defaults

The Tesla search found the right passage in both implementations, yet the result lists had different lengths. We will now compare all 30 development questions to see what matches and what changed.

This is a **parity check**: run two implementations on the same inputs and compare the behaviour we want to preserve. Here we care about which chunks are returned, their order, and how much evidence they would give the answer step.

### Start with the known-source hit count

Lab 01's **known-source hit@3** counts a question as a hit when at least one of the first three chunks comes from a paragraph named in `known_support`.

For example, if the labelled support is `S03-P00`, a returned chunk whose `source_id` is `S03-P00` counts as a hit. We compare source IDs because the label names a paragraph, while retrieval returns chunks of it.

The cell below uses our existing `hit_at_k()` and a small `is_hit()` function for LangChain documents. Notice that `is_hit()` checks every document it receives. The printed label alone cannot establish which `k` was actually used; we will inspect the returned lists next.


In [15]:
# plain_hits: True or False per question from your pipeline; lc_results: the store's hits for each question
from shared.search import hit_at_k

questions = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
plain_hits = hit_at_k(questions, corpus, vectors, settings)

def is_hit(question, hits):
    return any(doc.metadata["source_id"] in question["known_support"] for doc, _ in hits)

lc_results = [store.similarity_search_with_score(q["question"]) for q in questions]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
print(f"hit@3: plain {sum(plain_hits)} of {len(questions)} | LangChain {lc_hits} of {len(questions)}")

hit@3: plain 30 of 30 | LangChain 30 of 30


### What did we count?

1. `questions` loads the 30 development records, including each question's `known_support` paragraph IDs.
2. `hit_at_k()` returns one `True` or `False` per question for our plain search, using its default `k=3`.
3. `lc_results` keeps the LangChain search results for those same questions, initially without an explicit `k`.
4. `is_hit()` reads each returned document's `metadata["source_id"]` and checks whether any is labelled support.
5. `sum(...)` counts successful questions because Python counts `True` as 1.

The reference run prints 30 of 30 for both. That tells us each returned result set contains a known source. It does not tell us that both sets have the same length, contain the same chunks, or use the same order.

### Compare the actual chunk lists

Next, we will extract the chunk IDs from each search result. List equality (`a == b`) requires the same length and the same values in the same order.

For example, `["H02", "H01", "H05"]` differs from `["H02", "H01", "H05", "H04"]`, even though both contain H02. Predict how many complete lists will match before you run the cell.


In [16]:
# plain_ids, lc_ids: the chunk IDs each pipeline returned, one list per question, best first
plain_results = [search(q["question"], corpus, vectors, settings) for q in questions]
plain_ids = [[chunk["chunk_id"] for _, chunk in hits] for hits in plain_results]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
same = sum(a == b for a, b in zip(plain_ids, lc_ids))
print(f"same chunks in the same order: {same} of {len(questions)}")
print("first question | plain:", plain_ids[0], "| LangChain:", lc_ids[0])

same chunks in the same order: 0 of 30
first question | plain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00'] | LangChain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00', 'S00-P01-C00']


In the reference comparison, **0 of 30 complete lists match**. The first question makes the reason visible: LangChain returns a fourth chunk.

This does not mean all the retrieved evidence is different. A matching three-chunk prefix and an extra fourth result are enough to make list equality return `False` for every question.

Where did the fourth result come from? A **default** is the value a function uses when we omit an argument. `inspect.signature(...)` prints a function's parameters and their defaults, so we can check rather than guess.


In [17]:
# the arguments of the store's search method, with their default values
print(inspect.signature(store.similarity_search_with_score))

(query: 'str', k: 'int' = 4, **kwargs: 'Any') -> 'list[tuple[Document, float]]'


The signature shows `k: int = 4`. LangChain returns four documents unless we provide another value; our `search()` defaults to three.

This also explains a misleading label in the earlier output. It printed `hit@3` for both searches, but `is_hit()` checked all four LangChain results. That side was actually a **hit@4** count. We must align `k` before calling the metrics equivalent.

### What would the extra chunk cost?

If we send every returned chunk to the answer model, four chunks mean more evidence text than three. We will count the document-content tokens over all 30 questions, first using all four hits, then only the first three.

`evidence_tokens(hits)` tokenizes each document's `page_content` with the configured encoding and adds the lengths. This is a comparison of retrieved content size. It excludes citation labels, question text, instructions and request overhead, so it is not the complete billed input count. No chat call is made by this cell.


In [18]:
# four, three: the tokens in the chunks returned for all 30 questions, with four chunks and with the first three
import tiktoken

encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])

def evidence_tokens(hits):
    return sum(len(encoding.encode(doc.page_content)) for doc, _ in hits)

four = sum(evidence_tokens(hits) for hits in lc_results)
three = sum(evidence_tokens(hits[:3]) for hits in lc_results)
print(f"evidence tokens over 30 questions: k=4 {four} | k=3 {three} | {four / three - 1:.0%} more")

evidence tokens over 30 questions: k=4 9946 | k=3 7764 | 28% more


The reference totals are **9,946 content tokens with four chunks** and **7,764 with three**. `four / three - 1` gives the relative increase, about **28%**.

The overall hit count hid both the extra result and the extra evidence text. It also compared different values of `k`.

Now pass `k=3` explicitly, rerun the LangChain searches, and compare both the known-source count and the complete ID lists.


In [19]:
# lc_results, lc_ids: the store's hits again, now with k=3; same: the questions whose two lists match
lc_results = [store.similarity_search_with_score(q["question"], k=3) for q in questions]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
same = sum(a == b for a, b in zip(plain_ids, lc_ids))
print(f"hit@3: LangChain {lc_hits} of {len(questions)} | same chunks in the same order: {same} of {len(questions)}")

hit@3: LangChain 30 of 30 | same chunks in the same order: 30 of 30


With `k=3`, the reference run finds a labelled source for all 30 questions, and all 30 ordered chunk lists match our plain search.

We have aligned the number of results. Finally, compare the scores at corresponding ranks. The nested loops take one question's two result lists, then compare the first scores, the second scores, and the third scores.

`gap = max(gap, abs(plain_score - lc_score))` keeps the largest difference seen so far. It measures numerical agreement; the ID-list comparison above already checked that these scores refer to matching chunks.


In [20]:
# gap: the largest difference between your score and the store's score at the same rank of the same question
from shared.settings import embed_url

gap = 0
for plain, lc in zip(plain_results, lc_results):
    for (plain_score, _), (_, lc_score) in zip(plain, lc):
        gap = max(gap, abs(plain_score - lc_score))
print(f"largest score difference: {gap:.4f}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

largest score difference: 0.0021

text-embedding-3-small at https://api.openai.com/v1


The largest score difference in the reference run was **0.0021**. Small differences can occur because the store and our cached search obtained their document embeddings in separate calls.

### What have we established?

After setting `k=3`, both searches return the same chunk IDs in the same order for these 30 questions, with very similar scores. This gives us evidence that the LangChain store reproduces our retrieval behaviour on this set.

The initial total was insufficient: it concealed a default, and the printed metric name did not reflect the documents actually checked. When comparing implementations, inspect representative results and make the settings that affect the comparison explicit.

### Checkpoint D

| Comparison | Reference run |
|---|---|
| Initial source-hit count | 30 of 30 for each; plain uses three hits, LangChain uses four |
| Initial complete ID lists | 0 of 30 equal; LangChain's default is `k=4` |
| Retrieved-content tokens | 9,946 with four chunks; 7,764 with three |
| After setting `k=3` | Both hit@3 counts are 30 of 30; ordered lists match for 30 of 30 |
| Score difference after alignment | Largest difference 0.0021; small changes are possible |

Next, we will use LangChain to generate a cited answer. Matching retrieval does not automatically establish that the answer step behaves the same way.


---

## Part E: Generate Cited Answers as Structured Python Objects (self-study)

Our LangChain search now returns the same three chunks as our plain search. We will use those chunks to rebuild Lab 02's answer step: send rules and evidence to the chat model, request an answer with claims and quotes, and check the result.

We will build this in stages:

1. Call a chat model and inspect its reply object.
2. Describe the answer's fields with Pydantic classes.
3. Ask LangChain to return an object with that structure.
4. Check its quoted evidence with the same `quote_found()` function as before.

First, construct `ChatOpenAI` from our settings and send a simple question about vector stores. This call checks the model interface; it does not yet include retrieved evidence or a requested citation structure.


In [21]:
# chat: a LangChain chat model built by hand from the settings; reply: its answer to one message
import os
from langchain_openai import ChatOpenAI
from shared.settings import chat_url

chat = ChatOpenAI(model=settings["CHAT_MODEL"], base_url=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                  use_responses_api=False)
reply = chat.invoke("In one sentence, what does a vector store do?")
print(type(reply).__name__, "|", reply.content)
print(reply.usage_metadata)

AIMessage | A vector store is a database that efficiently stores, retrieves, and manages high-dimensional vectors, often used in machine learning and natural language processing for tasks like similarity search and recommendation systems.
{'input_tokens': 18, 'output_tokens': 36, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


`chat.invoke(...)` makes the model call and returns an **`AIMessage`**, LangChain's representation of an assistant reply.

| Value | What it contains | Earlier SDK equivalent |
|---|---|---|
| `reply.content` | The generated text in this call | `reply.choices[0].message.content` |
| `reply.usage_metadata` | Reported input, output and total token usage | `reply.usage` |

This first constructor sets the model, endpoint and key. `use_responses_api=False` keeps it on the Chat Completions route used by these labs.

### Keep configuration consistent with our plain pipeline

The hand-built object did not set the generation options selected by our shared settings helper. A comparison is easier to interpret when both pipelines use the same intended model configuration.

`lc_chat_model()` applies the settings through the same `chat_options()` logic used by our plain pipeline, including the configured temperature or reasoning options. Print its source, then use it to replace the hand-built object for the remaining calls.


In [22]:
# chat: the same model built by lc_chat_model(), with the options chat_options() picks for it
from shared.settings import lc_chat_model

print(inspect.getsource(lc_chat_model))
chat = lc_chat_model("CHAT_MODEL", settings)
print("model:", chat.model_name, "| temperature:", chat.temperature, "| reasoning_effort:", chat.reasoning_effort)

def lc_chat_model(name: str = "CHAT_MODEL", settings: dict | None = None, temperature: float = 0):
    """The same chat model as chat_client() + chat_options(), as a LangChain ChatOpenAI (Lab 05 on).

    use_responses_api=False keeps it on Chat Completions, which every compatible server implements.
    """
    from langchain_openai import ChatOpenAI
    settings = settings or load_settings()
    options = chat_options(name, settings, temperature)
    return ChatOpenAI(base_url=chat_url(settings), api_key=os.environ.get("OPENAI_API_KEY"),
                      use_responses_api=False, **options)

model: gpt-4o-mini | temperature: 0.0 | reasoning_effort: None


The reference configuration prints `temperature: 0.0` and `reasoning_effort: None`. Your values follow the selected model and settings. We will use `lc_chat_model()` for later labs so this configuration stays in one place.

### Describe the answer we want to receive

In Lab 02, `SCHEMA` was a JSON-schema dictionary describing an answer and its list of claims. `parse()` converted the model's JSON string into a dictionary and checked its shape.

Here, we will describe that structure with **Pydantic**: a library that builds and validates Python objects from typed fields.

- `Claim` describes one claim: its text, `source_id` and supporting quote, all strings.
- `CitedAnswer` describes the complete answer: an answer string and a list of `Claim` objects.
- `BaseModel` is the Pydantic base class that provides validation and conversion methods.

For example, `claims: list[Claim]` says that the field holds a list whose items follow the `Claim` structure. These fields have no defaults, so they are required.

We will define the classes, ask Pydantic to generate their JSON schema with `model_json_schema()`, then compare its required fields with Lab 02's schema. This checks the expected shape; it does not check whether an answer or quote is true.


In [23]:
# Claim, CitedAnswer: the answer's shape as Pydantic classes; schema: the JSON schema Pydantic writes from them
from pydantic import BaseModel
from shared import answer as answering

class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str

class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]

schema = CitedAnswer.model_json_schema()
print("answer fields: Pydantic", schema["required"], "| Lab 02", answering.SCHEMA["required"])
print("claim fields:  Pydantic", schema["$defs"]["Claim"]["required"],
      "| Lab 02", answering.SCHEMA["properties"]["claims"]["items"]["required"])

answer fields: Pydantic ['answer', 'claims'] | Lab 02 ['answer', 'claims']
claim fields:  Pydantic ['claim', 'source_id', 'quote'] | Lab 02 ['claim', 'source_id', 'quote']


Both representations require an `answer` and `claims`, and each claim requires `claim`, `source_id` and `quote`.

### Why inspect the generated schema?

Pydantic converts our Python field definitions into a JSON-schema representation. When using schema-based structured output, the provider receives that representation as part of the request.

Matching required fields does not mean the complete schemas are identical. For example, the handwritten schema explicitly forbids extra fields. Pydantic's default model behaviour and the provider's strict-schema handling are separate settings; we should not infer that every validation rule matches just because the required-field lists do.

The next cell prints the top-level keys and the definition of the `answer` field so we can see a concrete difference in representation.


In [24]:
# the top-level keys of each schema, and the "answer" field as each schema writes it
print("Pydantic keys:", list(schema))
print("Lab 02 keys:  ", list(answering.SCHEMA))
print("Pydantic answer field:", schema["properties"]["answer"])
print("Lab 02 answer field:  ", answering.SCHEMA["properties"]["answer"])

Pydantic keys: ['$defs', 'properties', 'required', 'title', 'type']
Lab 02 keys:   ['type', 'properties', 'required', 'additionalProperties']
Pydantic answer field: {'title': 'Answer', 'type': 'string'}
Lab 02 answer field:   {'type': 'string'}


The generated schema includes `$defs`, which stores the reusable definition of `Claim`, and `title` fields. For example, Pydantic's answer-field definition includes `{'title': 'Answer', 'type': 'string'}`, while Lab 02's is `{'type': 'string'}`.

The required information is similar, but the request representation differs. This is one reason we should compare actual replies rather than assume that replacing the schema implementation preserves every behaviour.

### Prepare the evidence the model will read

We will keep Lab 02's `[ID] text` format. For each `(Document, score)` result, `evidence()` uses `doc.id` for the citation label and `doc.page_content` for the text. The score is not included.

Unlike `pack()` from Lab 02, this function does not enforce a token budget. We first request exactly three documents, then format all three as evidence lines. Part H will compare this with the earlier packing approach.


In [25]:
# evidence(): the hits as "[ID] text" lines; lines: the Tesla question's three lines
def evidence(hits):
    return [f"[{doc.id}] {doc.page_content}" for doc, _ in hits]

lines = evidence(store.similarity_search_with_score(question, k=3))
for line in lines:
    print(line[:90], "...")

[S03-P00-C00] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) ...
[S03-P01-C00] Tesla gained experience in telephony and electrical engineering before emigr ...
[S03-P02-C00] Tesla went on to pursue his ideas of wireless lighting and electricity distr ...


The printed evidence lines identify the three Tesla chunks and show the start of their text. These are the lines both the model and our quote checker will read.

Now call `chat.with_structured_output(CitedAnswer, method=method)`. This configures a model-and-parser object that will return a `CitedAnswer` when invoked; creating it does not send a question yet.

Our `JSON_MODE` setting selects the request method:

- `json_schema` requests provider-enforced schema output where supported.
- `json_mode` requests valid JSON; our instructions describe the shape, and the parser validates it afterwards.

Both paths still need checks of the answer's meaning and evidence. A correctly shaped reply can contain an unsupported claim.


In [26]:
# structured: the chat model with CitedAnswer as its reply format; its replies come back as CitedAnswer objects
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
structured = chat.with_structured_output(CitedAnswer, method=method)
print(type(structured).__name__, "| method:", method)

RunnableSequence | method: json_schema


The reference configuration prints **`RunnableSequence`**: LangChain's object for running connected steps in order. Here, those steps make the model call and parse the response into the requested Pydantic object.

### Send the question and evidence

`messages_for()` constructs the same two message roles used in Lab 02:

- The **system** message contains `answering.RULES`, including the citation and abstention instructions.
- The **user** message contains the question and the evidence lines.

`structured.invoke(...)` sends those messages and returns a `CitedAnswer`. We can then read `reply.answer` and loop through `reply.claims`.

Our existing `quote_found()` expects a dictionary. Each Pydantic claim provides `model_dump()` to convert its fields into that format, so we can reuse the check without changing it.


In [27]:
# messages_for(): Lab 02's system and user messages; reply: the Tesla answer as a CitedAnswer object
def messages_for(question, lines):
    return [("system", answering.RULES), ("user", f"Question: {question}\nEvidence:\n" + "\n".join(lines))]

reply = structured.invoke(messages_for(question, lines))
print(type(reply).__name__, "|", reply.answer)
for claim in reply.claims:
    print(f"  quote found: {answering.quote_found(claim.model_dump(), lines)} | {claim.source_id} | {claim.quote}")

CitedAnswer | Nikola Tesla was born in the year 1856.
  quote found: True | S03-P00-C00 | 10 July 1856


The reference answer says that Tesla was born in **1856**, with a claim quoting *10 July 1856* from `S03-P00-C00`. The quote check returns `True`.

We now have a structured Python object where Lab 02 returned a dictionary. LangChain handles the structured-output request and parsing, while our application still checks the quoted evidence.

__What has validation established?__ The reply follows the expected structure and its reported quote was found in the evidence. This does not establish that the quote supports every part of the answer; Lab 02 showed why factual support needs additional judgement.

### Checkpoint E

| Step | Reference result |
|---|---|
| Plain chat call | `AIMessage`, with content and usage metadata |
| Shared chat configuration | `temperature: 0.0`, `reasoning_effort: None` for the reference model |
| Required schema fields | Same required answer and claim fields as Lab 02 |
| Structured wrapper | `RunnableSequence`, using `json_schema` |
| Cited answer | Tesla's birth year is 1856; the quote is found |

Next, we will test the answer steps on the full development set, keeping the evidence identical for both.


---

## Part F: Compare Answer Behaviour Using the Same Evidence (self-study)

We have generated one cited answer through LangChain. Now we want to know how its answer step compares with Lab 02's across the 30 development questions.

### Keep the evidence fixed

For each question, the next cell retrieves three documents once and constructs one `q_lines` list. It gives that exact list to both answer steps:

- `answering.answer(...)`: our plain Python implementation, returning a dictionary.
- `structured.invoke(...)`: the LangChain implementation, returning a `CitedAnswer`, which `.model_dump()` converts to the same dictionary format.

This lets us compare answer behaviour without also changing which evidence each model call receives. The model and intended settings stay aligned, but the schema representation, parsing and retry behaviour can differ, as can the model's replies between calls.

These evidence lines are not necessarily the pack used in Lab 04, which searched five chunks and applied a 400-token budget. This part compares two answer implementations on shared inputs; Part H returns to the wider pipeline difference.


In [28]:
# all_lines: each question's evidence lines; plain_replies, lc_replies: one answer per question from each pipeline
INSUFFICIENT = "Insufficient evidence."
all_lines, plain_replies, lc_replies = [], [], []
for q in questions:
    q_lines = evidence(store.similarity_search_with_score(q["question"], k=3))
    all_lines.append(q_lines)
    plain_replies.append(answering.answer(q["question"], q_lines, settings))
    lc_replies.append(structured.invoke(messages_for(q["question"], q_lines)).model_dump())
print(len(plain_replies), "answers from each pipeline")

30 answers from each pipeline


We now have 30 replies from each implementation. `all_lines` keeps the evidence for each question, while `plain_replies` and `lc_replies` keep the corresponding answers in the same order.

### What will we compare?

| Check | Why it matters | What it does not establish |
|---|---|---|
| Refusals | These questions have labelled answers, so refusals need inspection | Zero refusals does not mean every answer is correct |
| Number of claims | Shows how the answer is divided into cited statements | More claims do not automatically mean a better answer |
| Quotes found | Checks that the reported quotes occur in their cited evidence lines | A real quote may still fail to support the claim |
| Same cited IDs | Shows whether the replies cite the same sources in the same order | Different citations may both support a correct answer |
| Same answer text | Detects any difference in wording | Different wording may express the same fact |

`summary()` will count the first three checks, and `cited()` extracts a reply's source IDs for comparison. Because both implementations now use dictionaries, the same functions can read both.


In [29]:
# summary(): refusals, claims and quotes found over 30 replies; cited(): the source IDs one reply cites, in order
def summary(replies):
    refused = sum(reply["answer"].strip() == INSUFFICIENT for reply in replies)
    claims = [(q_lines, claim) for q_lines, reply in zip(all_lines, replies) for claim in reply["claims"]]
    found = sum(answering.quote_found(claim, q_lines) for q_lines, claim in claims)
    return f"refused {refused} of {len(replies)} | claims {len(claims)} | quotes found {found}"

def cited(reply):
    return [claim["source_id"] for claim in reply["claims"]]

print("plain:    ", summary(plain_replies))
print("LangChain:", summary(lc_replies))
print("same cited IDs:", sum(cited(p) == cited(l) for p, l in zip(plain_replies, lc_replies)), "of 30",
      "| same answer text:", sum(p["answer"] == l["answer"] for p, l in zip(plain_replies, lc_replies)), "of 30")

plain:     refused 0 of 30 | claims 30 | quotes found 30
LangChain: refused 0 of 30 | claims 32 | quotes found 32
same cited IDs: 28 of 30 | same answer text: 28 of 30


### How the summary works

1. A reply counts as refused when its stripped answer text equals `"Insufficient evidence."`.
2. The nested comprehension pairs each claim with its own question's evidence lines. This prevents checking a quote against another question's sources.
3. `quote_found()` tests each pair, and the `True` results are added up.
4. `cited()` extracts the ordered source IDs; equality compares their values and order.

In the reference run, neither implementation refuses. All quotes are found: **30 claims from the plain implementation and 32 from LangChain**. The cited-ID lists match for 28 of 30 questions, and the answer text matches for 28 of 30.

These totals do not explain the differences, and the question with different wording need not be the one with different citations. The next cell prints every pair whose text or cited IDs differ, plus any unfound quotes. Read those examples before interpreting the totals.


In [30]:
# not_found(): the quotes of one reply that the check did not find in its lines; the loop prints every pair that differs
def not_found(reply, q_lines):
    return [claim["quote"] for claim in reply["claims"] if not answering.quote_found(claim, q_lines)]

for plain, lc, q_lines in zip(plain_replies, lc_replies, all_lines):
    if plain["answer"] != lc["answer"] or cited(plain) != cited(lc) or not_found(plain, q_lines) + not_found(lc, q_lines):
        print("plain:    ", cited(plain), plain["answer"], not_found(plain, q_lines) or "")
        print("LangChain:", cited(lc), lc["answer"], not_found(lc, q_lines) or "", "\n")

plain:     ['S01-P00-C01'] 1745 
LangChain: ['S01-P00-C01'] Casimir Pulaski was born in Warsaw in 1745.  

plain:     ['S02-P00-C00'] Normandy is located in France. 
LangChain: ['S02-P00-C00', 'S02-P01-C01'] Normandy is located in France.  

plain:     ['S02-P02-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking." 
LangChain: ['S02-P02-C00', 'S02-P00-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking."  

plain:     ['S03-P00-C00'] Nikola Tesla was born in the year 1856. 
LangChain: ['S03-P00-C00'] 1856  



In the reference run, the printed differences concern four questions:

- **Pulaski:** One answer gives *1745* and the other says *Casimir Pulaski was born in Warsaw in 1745.* Both state the birth year from the same source.
- **Normandy:** The answer text matches, but LangChain adds a citation to `S02-P01-C01`.
- **Normans:** The answer text matches, but LangChain adds a citation to `S02-P00-C00`.
- **Tesla:** One answer says *Nikola Tesla was born in the year 1856.* and the other gives *1856*, using the same source.

Your run may differ. Read each pair to distinguish a changed fact from extra wording or another citation.

### How much variation occurs without changing frameworks?

Even with the same messages and settings, the model can produce different replies on separate calls. To put our comparison in context, we need a **control**: repeat the plain answer step with the same 30 questions and the same evidence lines.

The next cell compares that second plain run with its first run. It helps us see how much wording changes without switching implementations. One control run gives context; it cannot isolate every cause of a difference.


In [31]:
# again: a second run of your answer step on the same lines, the control
again = [answering.answer(q["question"], q_lines, settings) for q, q_lines in zip(questions, all_lines)]
print("plain, second run:", summary(again))
print("same answer text as the first run:", sum(a["answer"] == p["answer"] for a, p in zip(again, plain_replies)), "of 30")

plain, second run: refused 0 of 30 | claims 30 | quotes found 30
same answer text as the first run: 30 of 30


The reference control repeats the same answer text for **30 of 30 questions**, with 30 claims and all quotes found. The two implementations match on 28 of 30 answer texts in that comparison.

Other runs can show more variation in the control. Treat these counts as observations from a run, rather than a guarantee that a framework preserves or changes a particular number of answers.

### What might change with the schema representation?

Part E showed that the handwritten and Pydantic-generated schemas have different text. A separate repeated test on the Von Miller question found that the Pydantic-schema replies shortened quotes with `...` more often: 11 of 20 calls, compared with 0 of 20 using Lab 02's schema, with the same messages and model settings.

That result illustrates why the schema is worth checking when behaviour changes. It does not mean every wording difference in this run was caused by the schema. Our current `quote_found()` accepts shortened quotes when their pieces occur in order in the cited line, so shortening alone need not fail the check.


### Check a question the documents cannot answer

The development questions all have labelled support. To check abstention as well, we will use Lab 02's Moon question:

> *Who was the first person to walk on the Moon?*

The corpus has no passage answering it. Even if the model knows the answer from training, our rules require it to answer from the supplied evidence or say *Insufficient evidence.*

We will retrieve one evidence pack, give it to both answer steps, and ask three times each. The cell prints each refusal count and the first answer as a preview.


In [32]:
# plain_moon, lc_moon: three answers each to a question the corpus cannot answer
moon = "Who was the first person to walk on the Moon?"
moon_lines = evidence(store.similarity_search_with_score(moon, k=3))
plain_moon = [answering.answer(moon, moon_lines, settings)["answer"] for _ in range(3)]
lc_moon = [structured.invoke(messages_for(moon, moon_lines)).answer for _ in range(3)]
print(f"plain     abstained {plain_moon.count(INSUFFICIENT)} of 3 | {plain_moon[0]}")
print(f"LangChain abstained {lc_moon.count(INSUFFICIENT)} of 3 | {lc_moon[0]}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

plain     abstained 3 of 3 | Insufficient evidence.
LangChain abstained 3 of 3 | Insufficient evidence.

gpt-4o-mini at https://api.openai.com/v1


Both implementations refuse the Moon question in **3 of 3 calls** in the reference run.

### Read the comparison by behaviour

| Check | Plain implementation | LangChain, after setting `k=3` |
|---|---|---|
| Known-source hit@3 | 30 of 30 | 30 of 30 |
| Ordered retrieval IDs | Reference lists | Same lists for 30 of 30 questions |
| Refusals on the answerable set | 0 of 30 | 0 of 30 |
| Claims / quotes found | 30 / 30 | 32 / 32 |
| Moon question refused | 3 of 3 | 3 of 3 |

These checks show matching retrieval and similar measured answer behaviour on this question set. They do not prove that all answers are correct or that future requests will behave identically.

We have replaced several pieces of code: dictionaries with `Document` objects, matrix search with a store, and handwritten JSON handling with a structured-output wrapper. The comparison also exposed choices we must keep track of: retrieval defaults, schema representation and model variability.

### Checkpoint F

| Comparison | Reference run |
|---|---|
| Answerable set | Zero refusals; every quote found in each implementation |
| Between implementations | Same cited IDs for 28 of 30; same answer text for 28 of 30 |
| Plain run against itself | Same answer text for 30 of 30 |
| Unanswerable question | Moon question refused 3 of 3 by each |

Next, we will use metadata to decide which product's documents are eligible for retrieval before generating an answer.


---

## Part G: Restrict Retrieval to the Customer's Product (self-study)

Similarity search uses the question's text. Our application may also know something about the person asking, such as which product their account uses. Part G shows how to use that information to restrict the evidence.

Suppose an Atlas customer asks:

> *How quickly do reset links expire?*

The question does not say Atlas. Our corpus contains reset policies for both Atlas and Boreal, so either product's policy may be relevant to those words.

First, search without supplying the customer's product and generate three answers. Predict whether the evidence and replies will mention Atlas alone or both products.


In [33]:
# reset_lines: the evidence lines for the question; the loop asks the LangChain answer step three times
reset = "How quickly do reset links expire?"
reset_lines = evidence(store.similarity_search_with_score(reset, k=3))
print("evidence:", [line[1:line.index("]")] for line in reset_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, reset_lines)).answer)

evidence: ['H02', 'H04', 'H01']


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


The reference search returns **H02, H04 and H01**. The three answers give both products' times: Atlas's 30 minutes and Boreal's 10 minutes.

Those statements may be true, but we want an answer scoped to this customer's Atlas account. The question's wording does not provide that scope; the application must supply it.

### Use metadata to choose eligible documents

A **metadata filter** checks information stored alongside each document. For this store, it is a function that receives a `Document` and returns:

- `True` when that document is eligible for this search.
- `False` when it should be excluded.

We kept each record's title in Part A. The Atlas policy titles start with `"Atlas"`, so `atlas_only()` checks `document.metadata["title"].startswith("Atlas")`.

The store applies this filter before selecting the most similar results. H04 is therefore excluded from the candidates rather than removed after it has occupied one of the top-three positions.

Passing `filter=atlas_only` gives the store the function itself; the store calls it on each document. We will rerun the same question and answer three times with the filtered evidence.


In [34]:
# atlas_only(): True for the Atlas policies; atlas_lines: the evidence once every other document is left out
def atlas_only(document):
    return document.metadata["title"].startswith("Atlas")

atlas_lines = evidence(store.similarity_search_with_score(reset, k=3, filter=atlas_only))
print("evidence:", [line[1:line.index("]")] for line in atlas_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, atlas_lines)).answer)

evidence: ['H02', 'H01', 'H05']


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


The filtered reference search returns **H02, H01 and H05**. H04 is absent, and all three replies give Atlas's **30 minutes** only.

### What did the filter change?

The question stayed the same. We changed which documents the search could choose from, using account information outside the question.

Filtering does not guarantee that every remaining document answers the question: H05 is an Atlas billing policy. The store still ranks the eligible documents by similarity and returns up to `k` results.

Matching a title prefix is enough for this small corpus. In a real application, we would usually store a dedicated product field and derive the filter from the customer's account. A filter also does not enforce itself merely because the metadata exists; the application must apply it on the search.

Lab 11 uses this mechanism for permissions. Here we have practised product scope, using documents that are all public.

### Checkpoint G

| Search | Reference evidence and replies |
|---|---|
| Without product filter | H02, H04, H01; both products' times in 3 of 3 replies |
| With `atlas_only` | H02, H01, H05; Atlas's 30 minutes only, 3 of 3 |

If the number of eligible documents is smaller than `k`, the store can return fewer than `k`. The optional challenge uses Boreal to demonstrate this.


---

## Part H: Reuse the LangChain Pipeline and Compare Its Evidence (self-study)

We have built and inspected the LangChain components individually. We will now use the same steps from a module so later labs can focus on evaluation and improvements without copying the pipeline into every notebook.

`shared/lc_pipeline.py` contains these reusable pieces:

| Function | Step it keeps |
|---|---|
| `load_corpus()` | Load the same 277 records; reuse Lab 04's corpus loader under this name |
| `to_documents()` | Convert each record into a `Document` |
| `build_store()` | Embed and store the documents |
| `retrieve()` | Search with an explicit default `k=3`, and an optional filter |
| `evidence()` | Format the retrieved documents as `[ID] text` lines |
| `answer()` | Send the rules, question and evidence; return a `CitedAnswer` |

Read the source below and connect each function to the part that built it. The module also includes the `Claim` and `CitedAnswer` classes. It calls our settings helpers so model configuration remains in one place.


In [35]:
# the whole module: the classes of Part E and one function for each step
import shared.lc_pipeline

print(inspect.getsource(shared.lc_pipeline))

"""The pipeline of Labs 01 and 02, rebuilt with LangChain in Lab 05: the baseline for the labs after it.

load_corpus() is Lab 04's corpus(): the 272 chunks and the five policies. to_documents() turns its records into
LangChain Documents. build_store() embeds them into an InMemoryVectorStore. retrieve() searches it, with k=3 by
default (the store's own default is 4). evidence() writes the hits as "[ID] text" lines, and answer() asks for a
CitedAnswer with Lab 02's rules, so shared.answer.quote_found() checks its claims unchanged.
"""
from __future__ import annotations

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from pydantic import BaseModel

from shared.answer import RULES
from shared.settings import lc_chat_model, lc_embeddings
from shared.trace import corpus as load_corpus


class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str


class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]


d

### Run the reusable pipeline

The next cell assembles the starting objects in one expression:

`load_corpus()` → `to_documents(...)` → `build_store(..., settings)`

This creates a new store and embeds the corpus again; it does not reuse the earlier `store` object.

For each question, the loop then retrieves documents, formats evidence, generates a cited answer and checks its quotes. We will try both Tesla's birth year and the Moon question so the module demonstrates an answer and an abstention.


In [36]:
# pipeline_store: a new store built by the module; each question goes through retrieve(), evidence() and answer()
from shared import lc_pipeline

pipeline_store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
for q in (question, moon):
    q_lines = lc_pipeline.evidence(lc_pipeline.retrieve(pipeline_store, q))
    reply = lc_pipeline.answer(q, q_lines, settings)
    print(reply.answer, "| quotes found:", [answering.quote_found(claim.model_dump(), q_lines) for claim in reply.claims])

Nikola Tesla was born in the year 1856. | quotes found: [True]


Insufficient evidence. | quotes found: []


The reference module run answers Tesla's question with **1856** and a quote that is found. It refuses the Moon question with *Insufficient evidence.* An empty `quotes found: []` list for that refusal means no claims were returned.

### Is the complete pipeline identical to Lab 04's?

Part F supplied the same evidence to both answer steps. The reusable module makes a different evidence-selection choice from the pipeline we traced earlier:

| Pipeline | Evidence construction |
|---|---|
| Labs 02 and 04 | Retrieve five hits, then `pack()` complete chunks until the 400-token evidence budget is reached |
| `lc_pipeline` | Retrieve three hits, then `evidence()` formats all three; no evidence-token budget |

Both use `[ID] text` lines, but the number of lines and their total size may differ. To understand later answer changes, we need to know which evidence actually reached the model.

The next cell counts lines and tokens for the 30 questions under both approaches. It makes retrieval calls, including query embeddings, but no chat calls. These token counts include the evidence labels and text, unlike Part D's document-content-only comparison.


In [37]:
# packed: pack()'s lines per question (k=5, 400 tokens); lc_lines: evidence()'s lines per question (k=3, no budget)
def tokens(lines):
    return sum(len(encoding.encode(line)) for line in lines)

packed = [answering.pack(search(q["question"], corpus, vectors, settings, k=5), settings) for q in questions]
lc_lines = [lc_pipeline.evidence(lc_pipeline.retrieve(pipeline_store, q["question"])) for q in questions]
for name, per_question in [("pack()", packed), ("evidence()", lc_lines)]:
    counts = [len(q_lines) for q_lines in per_question]
    print(f"{name:11} {sum(counts)} lines, {sum(tokens(q_lines) for q_lines in per_question)} tokens"
          f" | lines a question: {sorted(set(counts))}")
print("questions where pack() gives more lines:", sum(len(a) > len(b) for a, b in zip(packed, lc_lines)), "of", len(questions))

pack()      125 lines, 10761 tokens | lines a question: [3, 4, 5]
evidence()  90 lines, 8382 tokens | lines a question: [3]
questions where pack() gives more lines: 23 of 30


In the reference comparison:

- `pack()` produces **125 evidence lines and 10,761 tokens** across the 30 questions, with three to five lines per question.
- `evidence()` produces **90 lines and 8,382 tokens**, with three lines for each question.
- For **23 of 30 questions**, the earlier pipeline includes more evidence lines.

The first three retrieved IDs matched in Part D, and this corpus's packs include at least three lines in the reference run. The difference is the extra evidence provided by the budgeted five-hit approach.

### Why does this matter for later labs?

If a later answer differs from a Lab 04 answer, we should inspect the actual evidence before attributing the difference to LangChain. The framework, evidence-selection settings and model variability can all affect the comparison.

For an answer-step comparison, keep the evidence fixed as Part F did. For a complete-pipeline comparison, record the retrieval and evidence choices as well as the final answer.

The following labs use the functions in `shared/lc_pipeline.py` by name and keep `quote_found()` from Lab 02. This gives them a common starting pipeline to evaluate and improve.

### Checkpoint H

| Check | Reference run |
|---|---|
| Module answers | Tesla: 1856 with quote found; Moon: *Insufficient evidence.* |
| `pack()` evidence | 125 lines, 10,761 tokens; three to five lines per question |
| `evidence()` evidence | 90 lines, 8,382 tokens; three lines per question |
| More lines from `pack()` | 23 of 30 questions |


---

## What you can do now

- Represent existing chunks as LangChain `Document` objects while preserving text, source IDs and metadata.
- Configure embeddings and chat models through the shared settings helpers, and distinguish input token IDs from output embedding values.
- Store document embeddings in `InMemoryVectorStore` and read its search results and similarity scores.
- Compare retrieval question by question, align `k`, and check the defaults behind a reported metric.
- Generate cited answers as Pydantic objects, check their quotes, and compare answer behaviour using identical evidence and a repeat-run control.
- Restrict retrieval using a customer's product, and explain how the reusable pipeline's evidence differs from the earlier budgeted pack.


## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Check a retriever against your own search.** Much LangChain code searches through a **retriever**, an object with
   `invoke()` and `batch()` that returns documents without scores. `store.as_retriever()` makes one from the store.
   Does it return what your `search()` returns?

   - Under **Task 1**, make a retriever with `store.as_retriever()`. The cell runs it on the 30 questions and counts
     the questions whose chunk IDs match `plain_ids` from Part D, in order.
   - If they do not all match, find out why from what the cell prints and the
     [`as_retriever` reference](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore),
     then fix the retriever and run the cell again.

   **Check:** the count before and after your fix. Our worked solution got 0 of 30, then 30 of 30.

   **Explain:** which check in this lab would have missed the difference, and which one caught it?

2. **Optional: write a filter for another customer.** Complete `boreal_only` under **Task 2** for a customer who uses
   Boreal. The cell asks Part G's question and one the Boreal policy does not answer, *How are Boreal invoices sent?*

   **Check:** for each question, the evidence IDs and the answer. Our worked solution got `['H04']` and Boreal's 10 minutes (*Reset links expire after 10 minutes.*) for Part G's question, and `['H04']` and *Insufficient evidence.* for the invoices question.

   **Explain:** how many documents did the store return for each question, and why? What would the model have had to
   go on without the abstention line?

### Task 1: Compare a retriever with our search

A retriever wraps the store in an object with `invoke()` for one query and `batch()` for several. It returns `Document` objects without their similarity scores.

Set `retriever` using `store.as_retriever()`, then run the provided comparison. The cell prints how many ordered ID lists match `plain_ids`, how many chunks the first query returned, and the retriever's `search_kwargs`.

If the lists differ, use those printed settings and the reference linked above to decide what to set explicitly. Then rerun the cell and explain the change.


In [38]:
# retriever: your retriever from the store; the cell counts the questions whose IDs match plain_ids from Part D
retriever = None    # TODO: a retriever from store.as_retriever()

if retriever is None:
    print("⚠️ Make a retriever, then run the cell again.")
else:
    retriever_ids = [[doc.id for doc in docs] for docs in retriever.batch([q["question"] for q in questions])]
    print(f"same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, retriever_ids))} of {len(questions)}")
    print("chunks per question:", len(retriever_ids[0]), "| search_kwargs:", retriever.search_kwargs)

⚠️ Make a retriever, then run the cell again.


### Task 2: Restrict evidence to Boreal (optional)

Use Part G's `atlas_only()` as the pattern for `boreal_only()`: return a Boolean based on the document's title metadata.

The cell tests both a reset-link question and an invoice question using the filtered evidence. Read the IDs as well as the answers. Only H04 describes Boreal, so asking for `k=3` cannot create three eligible documents.

For the invoice question, inspect whether H04 contains enough evidence to answer. The model should follow the same abstention instruction we tested earlier.


In [39]:
# boreal_only(): True for the Boreal documents; the cell asks two questions with it as the filter
def boreal_only(document):
    # TODO: return True for documents about Boreal (see atlas_only in Part G)
    return None

if boreal_only(documents[-1]) is None:
    print("⚠️ Complete boreal_only, then run the cell again.")
else:
    for q in (reset, "How are Boreal invoices sent?"):
        q_lines = evidence(store.similarity_search_with_score(q, k=3, filter=boreal_only))
        print([line[1:line.index("]")] for line in q_lines], "|", structured.invoke(messages_for(q, q_lines)).answer)

⚠️ Complete boreal_only, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain_community'` | code from an older tutorial; these labs install only LangChain's core packages | Import the store from `langchain_core.vectorstores` and the models from `langchain_openai`, as this lab does |
| `OpenAIError: Missing credentials. Please pass an api_key ...` | no key in the environment when the model or embeddings object was built | Add the key as Lab 00's README shows for your route, run Config, then build the object again |
| `OpenAIInvalidRequestError: Error code: 400 ... 'temperature' does not support 0.0 with this model.` | `ChatOpenAI(temperature=0)` built by hand for a reasoning model such as `gpt-6-luna` | Build models with `lc_chat_model()`, and set `REASONING_EFFORT=none` for `gpt-6-luna` |
| `ValueError: Unsupported function` followed by a schema | Lab 02's `SCHEMA` dictionary passed to `with_structured_output`, which needs a name for a dictionary schema | Pass the `CitedAnswer` class, or `{"name": "cited_answer", "strict": True, "schema": SCHEMA}` |
| `KeyError: 'access'` during a filtered search | the filter reads a field only some documents have (only the policies have `access`) | Read it with `document.metadata.get("access")` |

## Resources

- [LangChain: build a semantic search engine](https://docs.langchain.com/oss/python/langchain/knowledge-base): documents,
  embeddings, vector stores and retrievers, the steps of Parts A to D
- [LangChain: models, structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output):
  `with_structured_output` with a Pydantic class, its methods, and `include_raw=True` for the token counts
- [LangChain reference: `InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore):
  every search method, its defaults and the filter argument
- [LangChain reference: `OpenAIEmbeddings`](https://reference.langchain.com/python/langchain-openai/embeddings/base/OpenAIEmbeddings):
  `check_embedding_ctx_length` and the settings for other OpenAI-compatible servers
- [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/): fields, validation and `model_dump()`

Next: [Lab 06](../06_Evaluation_Set/README.md), *How to build an evaluation set for RAG*. Back to [the lab list](../README.md).